In [ ]:
!pip install ultralytics faster-whisper better-profanity moviepy gradio

In [ ]:
import cv2
import numpy as np
import moviepy.editor as mp
from ultralytics import YOLO
from faster_whisper import WhisperModel
from better_profanity import profanity
import torch
import gradio as gr
import os

In [ ]:
model = YOLO("yolov8n.pt")

whisper_model = WhisperModel(
    "base",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

profanity.load_censor_words()

In [ ]:
def detect_skin(frame):

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower = np.array([0,48,80], dtype=np.uint8)
    upper = np.array([20,255,255], dtype=np.uint8)

    mask = cv2.inRange(hsv, lower, upper)

    mask = cv2.GaussianBlur(
        mask,
        (3,3),
        0
    )

    return mask

In [ ]:
def detect_skin(frame):

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower = np.array([0,48,80], dtype=np.uint8)
    upper = np.array([20,255,255], dtype=np.uint8)

    mask = cv2.inRange(hsv, lower, upper)

    mask = cv2.GaussianBlur(
        mask,
        (3,3),
        0
    )

    return mask

In [ ]:
def transcribe_audio(video_path):

    video = mp.VideoFileClip(video_path)

    audio_path = "temp_audio.wav"

    video.audio.write_audiofile(
        audio_path,
        verbose=False,
        logger=None
    )

    segments, _ = whisper_model.transcribe(
        audio_path
    )

    text = " ".join(
        segment.text
        for segment in segments
    )

    os.remove(audio_path)

    return text

In [ ]:
def detect_profanity(text):

    return profanity.contains_profanity(text)

In [ ]:
def blur_nsfw_content(
        input_video,
        output_video):

    cap = cv2.VideoCapture(input_video)

    fps = cap.get(
        cv2.CAP_PROP_FPS
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (width,height)
    )

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = yolo_model(frame)

        skin_mask = detect_skin(frame)

        for result in results:

            for box in result.boxes:

                cls = int(box.cls[0])

                if yolo_model.names[cls] == "person":

                    x1,y1,x2,y2 = (
                        box.xyxy[0]
                        .cpu()
                        .numpy()
                        .astype(int)
                    )

                    roi = frame[
                        y1:y2,
                        x1:x2
                    ]

                    roi_skin = skin_mask[
                        y1:y2,
                        x1:x2
                    ]

                    if np.mean(roi_skin) > 10:

                        blur = cv2.GaussianBlur(
                            roi,
                            (99,99),
                            30
                        )

                        frame[
                            y1:y2,
                            x1:x2
                        ] = np.where(
                            roi_skin[:,:,None] > 0,
                            blur,
                            roi
                        )

        writer.write(frame)

    cap.release()
    writer.release()

In [ ]:
def moderate_video(
        video_path,
        filter_type):

    output_video = "processed_video.mp4"

    report = []

    if filter_type in [
        "Profanity",
        "Both"
    ]:

        transcript = transcribe_audio(
            video_path
        )

        bad_words = detect_profanity(
            transcript
        )

        report.append(
            f"Profanity Detected: {bad_words}"
        )

    if filter_type in [
        "NSFW",
        "Both"
    ]:

        blur_nsfw_content(
            video_path,
            output_video
        )

        report.append(
            "NSFW Filtering Applied"
        )

    else:

        output_video = video_path

    return (
        output_video,
        "\n".join(report)
    )

In [ ]:
app = gr.Interface(

    fn=moderate_video,

    inputs=[

        gr.Video(
            label="Upload Video"
        ),

        gr.Radio(
            choices=[
                "NSFW",
                "Profanity",
                "Both"
            ],
            value="Both",
            label="Filtering Mode"
        )

    ],

    outputs=[

        gr.Video(
            label="Processed Video"
        ),

        gr.Textbox(
            label="Moderation Report"
        )

    ],

    title="AI Video Moderation System",

    description="""
Upload a video and select moderation type.

NSFW      -> Blur exposed content
Profanity -> Detect offensive language
Both      -> Apply both filters
"""
)

In [ ]:
app.launch( share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c3992461531851582b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^